# Experiment [3] Visual Dashboard: Grouped-Holdout Generalization

This notebook compares random split performance against grouped-holdout transfer. The core question is whether the apparent gains survive dataset shift.

In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts" / "[2]"
DATA = ROOT / "data"

PALETTE = {
    "surf_jepa_v0": "#2563eb",
    "raw_flattened": "#64748b",
    "raw_stats": "#f97316",
    "surf_jepa_v0_plus_raw_stats": "#16a34a",
    "large_dual_s2": "#2563eb",
    "large_default": "#7c3aed",
    "large_dual_s2_jepa": "#2563eb",
    "large_default_jepa": "#7c3aed",
    "presto": "#db2777",
    "olmoearth": "#059669",
}
PRIORITY_CONDITIONS = ["clean", "sensor_off_s2", "temporal_drop_50", "temporal_drop_70", "s2_off_tdrop50"]
HOLDOUT_ORDER = ["rwanda-ceo", "togo", "togo-eval", "ethiopia", "lem-brazil"]

pd.options.display.max_columns = 80
pd.options.display.max_rows = 80


def standardize_table(df):
    if df.empty:
        return df
    rename = {
        "balanced_acc": "balanced_accuracy",
        "bal_acc": "balanced_accuracy",
        "mean_auc": "auc",
        "mean_f1": "f1",
        "budget": "label_budget",
    }
    return df.rename(columns={k: v for k, v in rename.items() if k in df.columns and v not in df.columns})


def read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT) if path.is_absolute() and ROOT in path.parents else path}")
        return pd.DataFrame()
    return standardize_table(pd.read_csv(path, **kwargs))


def read_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def pct(x):
    return f"{100*x:.1f}%" if pd.notna(x) else ""


def scorecard(df, cols, title="Scorecard", by=None):
    if df.empty:
        print("No data for scorecard")
        return
    work = df.copy()
    if by:
        work = work.set_index(by)
    display(work[cols].style.format({c: "{:.4f}" for c in cols if pd.api.types.is_numeric_dtype(work[c])}).set_caption(title))


def bar_metric(df, x, y, color=None, title=None, barmode="group", text=True, height=460):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.bar(
        df,
        x=x,
        y=y,
        color=color,
        barmode=barmode,
        text_auto=".3f" if text else False,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="", xaxis_title="", yaxis_title=y)
    fig.show()
    return fig


def line_metric(df, x, y, color=None, facet_col=None, title=None, markers=True, height=520):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.line(
        df,
        x=x,
        y=y,
        color=color,
        facet_col=facet_col,
        markers=markers,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="")
    fig.show()
    return fig


def heatmap(df, x, y, z, title=None, height=520, color_continuous_scale="RdYlGn"):
    if df.empty:
        print("No data to plot")
        return None
    pivot = df.pivot_table(index=y, columns=x, values=z, aggfunc="mean")
    fig = px.imshow(
        pivot,
        text_auto=".3f",
        aspect="auto",
        color_continuous_scale=color_continuous_scale,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", xaxis_title=x, yaxis_title=y)
    fig.show()
    return fig


def priority_average(df, group_cols, metric_cols=("f1", "auc", "balanced_accuracy"), conditions=PRIORITY_CONDITIONS):
    if df.empty:
        return df
    work = df[df["condition"].isin(conditions)].copy() if "condition" in df else df.copy()
    work = work.rename(columns={"balanced_acc": "balanced_accuracy", "mean_auc": "auc", "mean_f1": "f1"})
    available = [c for c in metric_cols if c in work.columns]
    if not available:
        return work.groupby(group_cols, dropna=False).size().reset_index(name="n")
    return work.groupby(group_cols, dropna=False)[available].mean().reset_index()


def normalize_model_name(name):
    text = str(name)
    return text.replace("_seed42", "").replace("_seed7", "").replace("_seed11", "")

RUN_DIR = ARTIFACTS / "cropharvest_jepa_v2_confirm_generalization"
ANALYSIS = RUN_DIR / "analysis"

In [2]:
random_rank = read_csv(ANALYSIS / "random_config_ranking.csv")
grouped_rank = read_csv(ANALYSIS / "grouped_config_ranking.csv")
scorecard(random_rank, [c for c in random_rank.columns if c != "config"], "[3] Random split ranking", by="config")
scorecard(grouped_rank, [c for c in grouped_rank.columns if c != "config"], "[3] Grouped-holdout ranking", by="config")

,seeds,priority_f1,priority_f1_sd,s2_tdrop50_f1,s2_tdrop50_auc
config,,,,,
large_dual_s2,3.0000,0.7869,0.0082,0.7548,0.7653
large_default,3.0000,0.7861,0.0082,0.7544,0.7614
medium_default,5.0000,0.7829,0.0056,0.7507,0.7521
medium_dual_s2,5.0000,0.7824,0.0061,0.7500,0.7539
medium_hard_dual,5.0000,0.7823,0.0079,0.7505,0.7543
medium_no_robust,3.0000,0.7805,0.0083,0.7458,0.7402


,seeds,grouped_f1,grouped_f1_sd,grouped_auc,grouped_bal_acc
config,,,,,
large_dual_s2,3.0000,0.5078,0.0042,0.6889,0.6230
large_default,3.0000,0.5069,0.0090,0.6891,0.6193
medium_default,5.0000,0.5060,0.0146,0.6839,0.6171
medium_dual_s2,5.0000,0.5036,0.0121,0.6799,0.6158
medium_no_robust,3.0000,0.5031,0.0187,0.6801,0.6160
medium_hard_dual,5.0000,0.5019,0.0087,0.6793,0.6153


In [3]:
# Random-vs-grouped gap.
if not random_rank.empty and not grouped_rank.empty:
    r = random_rank.rename(columns={"priority_f1": "random_f1", "priority_auc": "random_auc"})
    g = grouped_rank.rename(columns={"grouped_f1": "grouped_f1", "grouped_auc": "grouped_auc"})
    merged = g.merge(r[["config"] + [c for c in ["random_f1", "random_auc"] if c in r]], on="config", how="left")
    if "random_f1" in merged:
        merged["f1_transfer_gap"] = merged["grouped_f1"] - merged["random_f1"]
    display(merged)
    if "f1_transfer_gap" in merged:
        bar_metric(merged.sort_values("f1_transfer_gap"), x="config", y="f1_transfer_gap", color="config", title="[3] Grouped minus random F1")

,config,seeds,grouped_f1,grouped_f1_sd,grouped_auc,grouped_bal_acc,random_f1,f1_transfer_gap
0,large_dual_s2,3,0.507806,0.004184,0.688899,0.623021,0.786860,-0.279053
1,large_default,3,0.506892,0.009030,0.689073,0.619305,0.786103,-0.279211
2,medium_default,5,0.506043,0.014593,0.683943,0.617100,0.782941,-0.276898
3,medium_dual_s2,5,0.503550,0.012081,0.679926,0.615826,0.782431,-0.278880
4,medium_no_robust,3,0.503091,0.018688,0.680064,0.616040,0.780472,-0.277381
5,medium_hard_dual,5,0.501931,0.008675,0.679251,0.615258,0.782327,-0.280397


In [4]:
holdout_f1 = read_csv(ANALYSIS / "grouped_holdout_f1.csv")
if not holdout_f1.empty:
    long = holdout_f1.melt(id_vars="config", var_name="holdout", value_name="f1")
    heatmap(long, x="holdout", y="config", z="f1", title="[3] F1 by grouped holdout", height=620)

In [5]:
cond = read_csv(ANALYSIS / "grouped_condition_summary.csv")
if not cond.empty:
    metric_col = "f1" if "f1" in cond else "mean_f1"
    heatmap(cond, x="condition", y="config", z=metric_col, title="[3] Grouped-holdout stress robustness", height=650)

In [6]:
# Drill down into grouped-holdout result files from each run.
frames = []
for p in sorted(RUN_DIR.glob("*_seed*")):
    candidates = [p / "grouped_holdout" / "grouped_holdout_probe_results.csv", p / "grouped_holdout_probe_results.csv"]
    for c in candidates:
        if c.exists():
            df = pd.read_csv(c)
            config, seed = p.name.rsplit("_seed", 1)
            df["config"] = config
            df["seed"] = int(seed)
            frames.append(df)
            break
grouped = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(grouped.shape)
grouped.head()

(625, 11)


,model,condition,label_budget,n_train_sub,n_test,f1,auc,balanced_accuracy,config,seed,holdout
0,surf_jepa_v0,clean,0.01,641,3591,0.432318,0.664716,0.575579,large_default,11,rwanda-ceo
1,surf_jepa_v0,clean,0.05,3205,3591,0.379617,0.651848,0.559653,large_default,11,rwanda-ceo
2,surf_jepa_v0,clean,0.10,6410,3591,0.383929,0.689570,0.574421,large_default,11,rwanda-ceo
3,surf_jepa_v0,clean,0.25,16025,3591,0.426513,0.707019,0.595109,large_default,11,rwanda-ceo
4,surf_jepa_v0,clean,1.00,64101,3591,0.422521,0.705802,0.598970,large_default,11,rwanda-ceo


In [7]:
if not grouped.empty and {"holdout", "condition", "label_budget", "f1"}.issubset(grouped.columns):
    curves = grouped[grouped["condition"].isin(PRIORITY_CONDITIONS)].groupby(["config", "holdout", "label_budget"])[["f1", "auc"]].mean().reset_index()
    line_metric(curves, x="label_budget", y="f1", color="config", facet_col="holdout", title="[3] Grouped transfer label efficiency", height=720)